# Legal RAG Hybrid Demo Notebook

Notebook nay tap trung vao:
- chunking van ban phap ly xuong muc diem (point-level)
- hybrid retrieval dense + sparse deu dung BGE-M3
- chay local CPU, dung cau hinh key tu file .env
- tai su dung module trong core/ (config, db, llm, nlp)
- co cell uoc tinh thoi gian pipeline tren CPU

In [1]:
%pip install -q -U qdrant-client datasets pandas scikit-learn accelerate python-dotenv openai ipywidgets sentence-transformers fastembed "FlagEmbedding==1.3.5" "transformers==4.48.3" "huggingface-hub<1.0" "tokenizers<0.22"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from qdrant_client import QdrantClient
import os

QDRANT_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
if not QDRANT_URL:
    raise ValueError("QDRANT_URL đang trống. Vui lòng cấu hình Qdrant Cloud URL trong .env")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
print(f"✅ Kết nối Qdrant Cloud thành công: {client.get_collections()}")


✅ Kết nối Qdrant Docker thành công: collections=[]


In [ ]:
# Cell 2: local setup (.env), Qdrant, va BGE-M3 cho ca dense + sparse
from __future__ import annotations

import json
import math
import os
import re
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import display
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

from core.config import settings
from core.db import get_qdrant_client

# Force CPU mode for notebook portability on local machines
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"
load_dotenv()

DENSE_MODEL_NAME = os.getenv("LEGAL_DENSE_MODEL", "BAAI/bge-m3")
COLLECTION_NAME = os.getenv("QDRANT_LOCAL_COLLECTION", settings.QDRANT_COLLECTION)
DEVICE = "cpu"
QDRANT_URL = os.getenv("QDRANT_URL", settings.QDRANT_URL)
if not QDRANT_URL:
    raise ValueError("QDRANT_URL is required for cloud retrieval mode.")

print(f"Embedding device: {DEVICE}")
print(f"BGE-M3 model: {DENSE_MODEL_NAME}")

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=os.getenv("QDRANT_API_KEY", None),
)
print(f"Qdrant connected to: {QDRANT_URL}")


class LocalBGEHybridEncoder:
    """Use one BGE-M3 model to produce both dense and sparse vectors."""

    def __init__(self, model_name: str = "BAAI/bge-m3", device: str = "cpu"):
        self.model_name = model_name
        self.device = device
        self.model = BGEM3FlagModel(model_name, use_fp16=False, device=device)

    @staticmethod
    def _to_sparse_vector(weights: Dict[str, float]) -> models.SparseVector:
        if not weights:
            return models.SparseVector(indices=[], values=[])
        pairs = [(int(k), float(v)) for k, v in weights.items() if float(v) != 0.0]
        pairs.sort(key=lambda x: x[0])
        return models.SparseVector(
            indices=[idx for idx, _ in pairs],
            values=[val for _, val in pairs],
        )

    def encode_dense(self, texts: List[str], batch_size: int = 8) -> List[List[float]]:
        if isinstance(texts, str):
            texts = [texts]
        outputs = self.model.encode(
            texts,
            batch_size=batch_size,
            max_length=8192,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        dense_vecs = outputs["dense_vecs"]
        return dense_vecs.tolist() if hasattr(dense_vecs, "tolist") else [list(vec) for vec in dense_vecs]

    def encode_sparse_documents(self, texts: List[str], batch_size: int = 8) -> List[models.SparseVector]:
        if isinstance(texts, str):
            texts = [texts]
        outputs = self.model.encode(
            texts,
            batch_size=batch_size,
            max_length=8192,
            return_dense=False,
            return_sparse=True,
            return_colbert_vecs=False,
        )
        lexical_weights = outputs["lexical_weights"]
        return [self._to_sparse_vector(weights) for weights in lexical_weights]

    def encode_query_sparse(self, text: str) -> models.SparseVector:
        return self.encode_sparse_documents([text], batch_size=1)[0]


print("Loading BGE-M3 for dense + sparse...")
hybrid_encoder = LocalBGEHybridEncoder(model_name=DENSE_MODEL_NAME, device=DEVICE)

probe_dense = hybrid_encoder.encode_dense(["kiem tra kich thuoc vector"], batch_size=1)[0]
dense_dim = len(probe_dense)
print(f"Dense embedding dimension: {dense_dim}")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(
                size=dense_dim,
                distance=models.Distance.COSINE,
                on_disk=True,
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False)
            )
        },
    )
    print(f"Created Qdrant collection: {COLLECTION_NAME}")
else:
    print(f"Qdrant collection already exists: {COLLECTION_NAME}")

# Keep old variable names for compatibility with later cells
embedder = hybrid_encoder
sparse_encoder = hybrid_encoder

print("Local BGE-M3 hybrid encoder is ready (dense + sparse).")

Embedding device: cpu
BGE-M3 model: BAAI/bge-m3


C:\Users\nhn26\AppData\Local\Temp\ipykernel_65008\1365372347.py:34: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


Qdrant connected to: https://your-cluster.cloud.qdrant.io:6333
Loading BGE-M3 for dense + sparse...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

: 

In [ ]:
# Cell 3: tien xu ly full dataset + lay 10000 docs tu top 200 sector (moi sector 25 docs)
print("Dang load metadata/content tu Hugging Face dataset ...")
ds_metadata = load_dataset("th1nhng0/vietnamese-legal-documents", "metadata", split="data")
ds_content = load_dataset("th1nhng0/vietnamese-legal-documents", "content", split="data")

from bisect import bisect_left


def split_sector_list(value: Any) -> List[str]:
    """
    Tach legal_sectors thanh mang sector da sap xep tang dan, unique.
    Muc tieu: ho tro tra cuu O(log n) bang binary search.
    """
    if value is None:
        return []

    if isinstance(value, list):
        raw_text = ", ".join(str(x) for x in value if str(x).strip())
    else:
        raw_text = str(value).strip()

    if not raw_text or raw_text.lower() == "nan":
        return []

    # Chi tach theo " - " de tranh cat nham cac tu co dau '-' ben trong
    parts = re.split(r"\s+-\s+|\s*,\s*", raw_text)
    sectors = [part.strip() for part in parts if part and part.strip()]

    # Sap xep tang dan + unique de tra binary search
    return sorted(set(sectors))


def contains_sector(sorted_sectors: List[str], target: str) -> bool:
    if not sorted_sectors or not target:
        return False
    pos = bisect_left(sorted_sectors, target)
    return pos < len(sorted_sectors) and sorted_sectors[pos] == target


# Backward-compatible alias cho cac cell duoi
parse_sector_list = split_sector_list


df_meta = ds_metadata.to_pandas().copy()
df_meta["id"] = df_meta["id"].astype(str)
df_meta["sector_list"] = df_meta["legal_sectors"].apply(split_sector_list)
df_meta = df_meta[df_meta["sector_list"].map(len) > 0].copy()

# Kiem tra duplicate metadata id
dup_meta_ids = int(df_meta["id"].duplicated().sum())
if dup_meta_ids > 0:
    print(f"Canh bao: co {dup_meta_ids} id bi trung trong metadata; giu ban ghi dau tien.")
    df_meta = df_meta.drop_duplicates(subset="id", keep="first").copy()

sector_counter = (
    df_meta[["id", "sector_list"]]
    .explode("sector_list")
    .rename(columns={"sector_list": "sector"})
)

TOP_K_SECTORS = 200
PER_SECTOR_DOCS = 50
target_docs = TOP_K_SECTORS * PER_SECTOR_DOCS
seed = 42

hot_sector_counts = sector_counter["sector"].value_counts().head(TOP_K_SECTORS)
HOT_SECTORS = hot_sector_counts.index.tolist()

print(f"Top {TOP_K_SECTORS} sector nhieu nhat:")
display(hot_sector_counts.to_frame("document_count"))

# Tao mapping sector -> tap doc_id co chua sector do
sector_to_doc_ids: Dict[str, List[str]] = {}
for sector in HOT_SECTORS:
    ids = (
        df_meta.loc[df_meta["sector_list"].apply(lambda arr: contains_sector(arr, sector)), "id"]
        .drop_duplicates()
        .sample(frac=1, random_state=seed)
        .tolist()
    )
    sector_to_doc_ids[sector] = ids

# Moi sector lay toi da 50 docs
selected_ids = []
for sector in HOT_SECTORS:
    selected_ids.extend(sector_to_doc_ids[sector][:PER_SECTOR_DOCS])

# Deduplicate doc id sau khi gop nhieu sector
selected_ids = list(dict.fromkeys(selected_ids))

# Neu chua du 10000 thi bo sung tu tap co it nhat 1 sector thuoc HOT_SECTORS
if len(selected_ids) < target_docs:
    remaining_ids = (
        df_meta.loc[
            df_meta["sector_list"].apply(lambda arr: any(contains_sector(arr, s) for s in HOT_SECTORS)),
            "id",
        ]
        .drop_duplicates()
        .tolist()
    )
    selected_set = set(selected_ids)
    remaining_ids = [doc_id for doc_id in remaining_ids if doc_id not in selected_set]
    if remaining_ids:
        fill_n = min(target_docs - len(selected_ids), len(remaining_ids))
        fill_ids = pd.Series(remaining_ids).sample(n=fill_n, random_state=seed).tolist()
        selected_ids.extend(fill_ids)

# Cat dung 10000 docs neu vuot
if len(selected_ids) > target_docs:
    selected_ids = pd.Series(selected_ids).sample(n=target_docs, random_state=seed).tolist()

# Giu ten bien df_500 de tuong thich cac cell sau, nhung so luong la 10000
df_500 = (
    df_meta[df_meta["id"].isin(set(selected_ids))]
    .drop_duplicates(subset="id")
    .sample(frac=1, random_state=seed)
    .reset_index(drop=True)
)
df_test = df_500.head(min(50, len(df_500))).copy()

valid_ids_500 = set(df_500["id"].astype(str))
valid_ids_test = set(df_test["id"].astype(str))

ds_metadata_500 = ds_metadata.filter(lambda row: str(row["id"]) in valid_ids_500)
ds_content_500 = ds_content.filter(lambda row: str(row["id"]) in valid_ids_500)
ds_metadata_test = ds_metadata.filter(lambda row: str(row["id"]) in valid_ids_test)
ds_content_test = ds_content.filter(lambda row: str(row["id"]) in valid_ids_test)

metadata_by_id = {str(row["id"]): row for row in ds_metadata_500}
content_by_id = {str(row["id"]): row for row in ds_content_500}

print(f"So van ban da chon (muc tieu {target_docs}): {len(ds_content_500)}")
print(f"So van ban test tam de demo: {len(ds_content_test)}")

# Thong ke lai theo sector trong mau da chon
demo_distribution = (
    df_500.explode("sector_list")["sector_list"]
    .value_counts()
    .head(TOP_K_SECTORS)
)
print(f"Phan bo sector trong tap {len(df_500)} sample (top {TOP_K_SECTORS}):")
display(demo_distribution.to_frame("sample_count"))

In [ ]:
# Cell 4: in thu 1 van ban trong tap test de kiem tra
target_id = str(df_test.iloc[0]["id"])
metadata_raw = metadata_by_id[target_id]
content_raw = content_by_id[target_id]["content"]

print(f"Target test document id: {target_id}")
print("=" * 120)
print("METADATA")
print("=" * 120)
for key, value in metadata_raw.items():
    if key != "id":
        print(f"{key}: {value}")

print("\n" + "=" * 120)
print("RAW CONTENT")
print("=" * 120)
print(content_raw)

In [ ]:
# Cell 5: chunker phap ly nang cao -> tach can cu / dieu / khoan / diem / phu luc
def compact_whitespace(text: str) -> str:
    return re.sub(r"[ \t]+", " ", str(text or "")).strip()


class AdvancedLegalChunker:
    def __init__(self, appendix_chunk_size: int = 1000):
        self.appendix_chunk_size = appendix_chunk_size
        self.appendix_pattern = re.compile(r"(?im)^\s*(PHU LUC|PHỤ LỤC|DANH MUC|DANH MỤC)\b.*$")
        self.chapter_pattern = re.compile(r"(?im)^\s*(Chương\s+[IVXLCDM0-9]+)\s*(.*)$")
        self.article_pattern = re.compile(r"(?im)^\s*(Điều\s+\d+[A-Za-z0-9\/\-]*)[\.\:\-]?\s*(.*)$")
        self.clause_pattern = re.compile(
            r"(?im)^\s*(Khoản\s+\d+[\.\:\-]?)\s*(.*)$|^\s*(\d+[\.\)])\s*(.*)$"
        )
        self.point_pattern = re.compile(
            r"(?im)^[ \t]*(Điểm\s+[a-zđ]+[\.\:\-]?)\s*(.*)$|^[ \t]*([a-zđ][\.\)])\s*(.*)$"
        )

    def _normalize_metadata(self, metadata):
        data = dict(metadata)
        data["id"] = str(data.get("id", ""))
        data["legal_sectors_list"] = parse_sector_list(data.get("legal_sectors"))
        data["legal_sectors_text"] = "; ".join(data["legal_sectors_list"])
        data["signer"] = data.get("signer") or data.get("signers") or ""
        return data

    def _build_metadata_header(
        self,
        metadata,
        reference_tag,
        is_appendix,
        article_ref=None,
        clause_ref=None,
        chapter_ref=None,
        point_ref=None,
    ):
        header_lines = [
            "[LEGAL HEADER]",
            f"- Title: {metadata.get('title', 'N/A')}",
            f"- Document number: {metadata.get('document_number', 'N/A')}",
            f"- Legal type: {metadata.get('legal_type', 'N/A')}",
            f"- Issuing authority: {metadata.get('issuing_authority', 'N/A')}",
            f"- Legal sectors: {metadata.get('legal_sectors_text', metadata.get('legal_sectors', 'N/A'))}",
            f"- Reference tag: {reference_tag}",
            f"- Chapter: {chapter_ref or 'N/A'}",
            f"- Article: {article_ref or 'N/A'}",
            f"- Clause: {clause_ref or 'N/A'}",
            f"- Point: {point_ref or 'N/A'}",
            f"- Is appendix: {is_appendix}",
            "",
        ]
        return "\n".join(header_lines)

    def _build_citation(
        self,
        metadata,
        chapter_ref,
        article_ref,
        clause_ref,
        is_appendix,
        appendix_part=None,
        point_ref=None,
    ):
        pieces = [metadata.get("document_number") or metadata.get("title") or "Van ban"]
        if chapter_ref:
            pieces.append(chapter_ref)
        if article_ref:
            pieces.append(article_ref)
        if clause_ref:
            pieces.append(clause_ref)
        if point_ref:
            pieces.append(point_ref)
        if is_appendix and appendix_part:
            pieces.append(appendix_part)
        return " | ".join([piece for piece in pieces if piece])

    def _split_main_and_appendix(self, content):
        match = self.appendix_pattern.search(content)
        if match:
            return content[: match.start()].strip(), content[match.start() :].strip()
        return content.strip(), ""

    def _split_articles(self, main_text):
        intro_lines = []
        articles = []
        current_article = None
        current_chapter = None

        for raw_line in main_text.splitlines():
            line = raw_line.rstrip()

            if not line.strip():
                if current_article is None:
                    intro_lines.append("")
                else:
                    current_article["lines"].append("")
                continue

            chapter_match = self.chapter_pattern.match(line)
            if chapter_match:
                current_chapter = compact_whitespace(line)
                if current_article is None:
                    intro_lines.append(line)
                else:
                    current_article["lines"].append(line)
                continue

            article_match = self.article_pattern.match(line)
            if article_match:
                if current_article is not None:
                    articles.append(current_article)

                current_article = {
                    "chapter_ref": current_chapter,
                    "article_ref": compact_whitespace(article_match.group(1)),
                    "article_title": compact_whitespace(article_match.group(2)),
                    "lines": [line],
                }
                continue

            if current_article is None:
                intro_lines.append(line)
            else:
                current_article["lines"].append(line)

        if current_article is not None:
            articles.append(current_article)

        return "\n".join(intro_lines).strip(), articles

    def _split_clauses(self, article):
        full_article_text = "\n".join(article["lines"]).strip()
        body_lines = article["lines"][1:] if len(article["lines"]) > 1 else []

        clauses = []
        current_clause = None

        for raw_line in body_lines:
            line = raw_line.rstrip()
            clause_match = self.clause_pattern.match(line)

            if clause_match:
                if current_clause is not None:
                    clauses.append(current_clause)

                clause_ref = compact_whitespace(clause_match.group(1) or clause_match.group(3))
                clause_tail = compact_whitespace(clause_match.group(2) or clause_match.group(4))
                clause_line = f"{clause_ref} {clause_tail}".strip()
                current_clause = {
                    "clause_ref": clause_ref,
                    "lines": [clause_line],
                }
                continue

            if current_clause is None:
                if line.strip():
                    current_clause = {
                        "clause_ref": None,
                        "lines": [line],
                    }
            else:
                current_clause["lines"].append(line)

        if current_clause is not None:
            clauses.append(current_clause)

        if not clauses:
            clauses = [
                {
                    "clause_ref": None,
                    "lines": article["lines"],
                }
            ]

        return full_article_text, clauses

    def _split_points(self, clause):
        """Split a clause dict into a list of point dicts.

        Each point dict has keys ``point_ref`` and ``lines``.
        If the clause contains no recognisable points the whole clause body
        is returned as a single point with ``point_ref=None``.
        """
        body_lines = clause["lines"]

        points = []
        current_point = None

        for raw_line in body_lines:
            line = raw_line.rstrip()
            point_match = self.point_pattern.match(line)

            if point_match:
                if current_point is not None:
                    points.append(current_point)

                point_ref = compact_whitespace(point_match.group(1) or point_match.group(3))
                point_tail = compact_whitespace(point_match.group(2) or point_match.group(4))
                point_line = f"{point_ref} {point_tail}".strip()
                current_point = {
                    "point_ref": point_ref,
                    "lines": [point_line],
                }
                continue

            if current_point is None:
                if line.strip():
                    current_point = {
                        "point_ref": None,
                        "lines": [line],
                    }
            else:
                current_point["lines"].append(line)

        if current_point is not None:
            points.append(current_point)

        if not points:
            points = [
                {
                    "point_ref": None,
                    "lines": clause["lines"],
                }
            ]

        return points

    def _appendix_chunks(self, appendix_text, metadata, doc_id):
        chunks = []
        if not appendix_text:
            return chunks

        active_article_ref = "Phu luc"
        active_clause_ref = None
        chunk_article_ref = active_article_ref
        chunk_clause_ref = active_clause_ref

        current_lines = []
        current_len = 0
        part_idx = 1

        def flush_chunk():
            nonlocal current_lines, current_len, part_idx, chunk_article_ref, chunk_clause_ref
            if not current_lines:
                return

            current_chunk = "\n".join(current_lines).strip()
            header_app = self._build_metadata_header(
                metadata=metadata,
                reference_tag=f"Phu luc - P{part_idx}",
                is_appendix=True,
                article_ref=chunk_article_ref,
                clause_ref=chunk_clause_ref,
            )
            citation = self._build_citation(
                metadata=metadata,
                chapter_ref=None,
                article_ref=chunk_article_ref,
                clause_ref=chunk_clause_ref,
                is_appendix=True,
                appendix_part=f"P{part_idx}",
            )

            chunks.append(
                {
                    "chunk_id": f"{doc_id}::appendix::{part_idx}",
                    "text_to_embed": f"{header_app}[PHAN {part_idx}]\n{current_chunk.strip()}",
                    "reference_tag": f"Phu luc - P{part_idx}",
                    "unit_text": current_chunk.strip(),
                    "metadata": {
                        **metadata,
                        "is_appendix": True,
                        "chapter_ref": None,
                        "article_id": f"{doc_id}::appendix::{part_idx}",
                        "article_ref": chunk_article_ref,
                        "article_title": "Phu luc",
                        "clause_ref": chunk_clause_ref or f"P{part_idx}",
                        "point_ref": None,
                        "parent_article_text": current_chunk.strip(),
                        "reference_citation": citation,
                    },
                }
            )
            current_lines = []
            current_len = 0
            part_idx += 1
            chunk_article_ref = active_article_ref
            chunk_clause_ref = active_clause_ref

        for raw_line in appendix_text.splitlines():
            line = raw_line.rstrip()

            article_match = self.article_pattern.match(line)
            if article_match:
                active_article_ref = compact_whitespace(article_match.group(1))
                active_clause_ref = None

            clause_match = self.clause_pattern.match(line)
            if clause_match:
                active_clause_ref = compact_whitespace(clause_match.group(1) or clause_match.group(3))

            candidate_len = len(line) + 1
            if current_lines and (current_len + candidate_len > self.appendix_chunk_size):
                flush_chunk()

            if not current_lines:
                chunk_article_ref = active_article_ref
                chunk_clause_ref = active_clause_ref

            current_lines.append(line)
            current_len += candidate_len
            chunk_clause_ref = active_clause_ref

        flush_chunk()
        return chunks

    def process_document(self, content, metadata):
        metadata = self._normalize_metadata(metadata)
        content = str(content or "").replace("\r\n", "\n").strip()
        doc_id = metadata.get("id") or str(uuid.uuid4())
        chunks = []

        main_text, appendix_text = self._split_main_and_appendix(content)
        intro_text, articles = self._split_articles(main_text)

        if intro_text:
            article_id = f"{doc_id}::preamble"
            ref_tag = "Can cu"
            citation = self._build_citation(
                metadata=metadata,
                chapter_ref=None,
                article_ref=ref_tag,
                clause_ref=None,
                is_appendix=False,
            )
            header = self._build_metadata_header(
                metadata=metadata,
                reference_tag=ref_tag,
                is_appendix=False,
                article_ref=ref_tag,
            )
            chunks.append(
                {
                    "chunk_id": f"{article_id}::0",
                    "text_to_embed": f"{header}[PHAN MO DAU / CAN CU]\n{intro_text}",
                    "reference_tag": ref_tag,
                    "unit_text": intro_text,
                    "metadata": {
                        **metadata,
                        "is_appendix": False,
                        "chapter_ref": None,
                        "article_id": article_id,
                        "article_ref": ref_tag,
                        "article_title": "Phan mo dau va can cu",
                        "clause_ref": None,
                        "point_ref": None,
                        "parent_article_text": intro_text,
                        "reference_citation": citation,
                    },
                }
            )

        for article_index, article in enumerate(articles, start=1):
            article_id = f"{doc_id}::article::{article_index}"
            full_article_text, clause_entries = self._split_clauses(article)
            article_context = (
                f"{article['chapter_ref']}\n{full_article_text}".strip()
                if article.get("chapter_ref")
                else full_article_text
            )

            for clause_index, clause in enumerate(clause_entries, start=1):
                clause_ref = clause["clause_ref"]
                point_entries = self._split_points(clause)

                for point_index, point in enumerate(point_entries, start=1):
                    point_ref = point["point_ref"]
                    unit_text = "\n".join(point["lines"]).strip()

                    # Build reference_tag: Article - Clause - Point
                    tag_parts = [article["article_ref"]]
                    if clause_ref is not None:
                        tag_parts.append(clause_ref)
                    if point_ref is not None:
                        tag_parts.append(point_ref)
                    reference_tag = " - ".join(tag_parts)

                    citation = self._build_citation(
                        metadata=metadata,
                        chapter_ref=article.get("chapter_ref"),
                        article_ref=article["article_ref"],
                        clause_ref=clause_ref,
                        is_appendix=False,
                        point_ref=point_ref,
                    )
                    header = self._build_metadata_header(
                        metadata=metadata,
                        reference_tag=reference_tag,
                        is_appendix=False,
                        article_ref=article["article_ref"],
                        clause_ref=clause_ref,
                        chapter_ref=article.get("chapter_ref"),
                        point_ref=point_ref,
                    )

                    chunks.append(
                        {
                            "chunk_id": f"{article_id}::clause::{clause_index}::point::{point_index}",
                            "text_to_embed": (
                                f"{header}[DIEU CHA]\n{article_context}\n\n"
                                f"[DIEM DUOC TRUY HOI]\n{unit_text}"
                            ),
                            "reference_tag": reference_tag,
                            "unit_text": unit_text,
                            "metadata": {
                                **metadata,
                                "is_appendix": False,
                                "chapter_ref": article.get("chapter_ref"),
                                "article_id": article_id,
                                "article_ref": article["article_ref"],
                                "article_title": article.get("article_title"),
                                "clause_ref": clause_ref,
                                "point_ref": point_ref,
                                "parent_article_text": article_context,
                                "reference_citation": citation,
                            },
                        }
                    )

        chunks.extend(self._appendix_chunks(appendix_text, metadata, doc_id))
        return chunks


chunker = AdvancedLegalChunker(appendix_chunk_size=1000)
result_chunks = chunker.process_document(content_raw, metadata_raw)

print(f"KIEM TRA document id: {target_id}")
print(f"Tong so chunk sinh ra: {len(result_chunks)}")
print("-" * 140)

for idx, chunk in enumerate(result_chunks, start=1):
    meta = chunk["metadata"]
    preview = chunk["text_to_embed"].replace("\n", " | ")
    print(
        f"[{idx:03d}] appendix={meta['is_appendix']} | "
        f"article_id={meta['article_id']} | "
        f"ref={chunk['reference_tag']} | "
        f"clause={meta.get('clause_ref')} | "
        f"point={meta.get('point_ref')} | "
        f"len={len(chunk['text_to_embed'])} | "
        f"{preview[:220]}..."
    )


In [ ]:
# Cell 5.1: In chi tiết cấu trúc (Anatomy) của các chunk đầu tiên
import json

print(f"=== CHI TIẾT KẾT QUẢ CHUNKING CHO VĂN BẢN: {target_id} ===")
print(f"Tổng số chunk được tạo ra: {len(result_chunks)}\n")

# Lấy 2 chunk đầu tiên để kiểm tra (Hoặc bạn có thể đổi thành result_chunks[0:1] nếu chỉ muốn xem chunk đầu)
chunks_to_inspect = result_chunks[:2]

for i, chunk in enumerate(chunks_to_inspect):
    print("=" * 100)
    print(f" CHUNK [{i}] - {chunk['reference_tag']}")
    print("=" * 100)
    
    # 1. Hiển thị Text to Embed (Dữ liệu sẽ bị biến thành mảng số Vector)
    print("\n[1] VĂN BẢN ĐỂ NHÚNG (Text to Embed - Vector model sẽ đọc cái này):")
    print("-" * 60)
    print(chunk["text_to_embed"])
    print("-" * 60)
    
    # 2. Hiển thị Unit Text (Đoạn text nhỏ nhất để match)
    print("\n[2] VĂN BẢN MẢNH (Unit Text / Child Chunk):")
    print("-" * 60)
    print(chunk["unit_text"])
    print("-" * 60)
    
    # 3. Hiển thị Parent Context (Đoạn text to chứa đoạn text nhỏ)
    print("\n[3] NGỮ CẢNH ĐIỀU LUẬT (Parent Article Text - Sẽ đưa cho LLM đọc):")
    print("-" * 60)
    print(chunk["metadata"].get("parent_article_text", "Không có"))
    print("-" * 60)

    # 4. Hiển thị Metadata sẽ được đẩy lên Qdrant Payload
    print("\n[4] METADATA (Sẽ đẩy lên Qdrant để Filter và Rerank):")
    print("-" * 60)
    # Trích xuất một số key quan trọng để in cho đỡ rối mắt
    meta = chunk['metadata']
    meta_preview = {
        "article_id": meta.get("article_id"),
        "chapter_ref": meta.get("chapter_ref"),
        "article_ref": meta.get("article_ref"),
        "clause_ref": meta.get("clause_ref"),
        "point_ref": meta.get("point_ref"),
        "reference_citation": meta.get("reference_citation"),
        "is_appendix": meta.get("is_appendix"),
        "legal_sectors_list": meta.get("legal_sectors_list")
    }
    print(json.dumps(meta_preview, indent=4, ensure_ascii=False))
    print("\n" + "*" * 100 + "\n")


In [ ]:
# Cell 6: config Qdrant tu .env, chunk docs va tao dense+sparse embeddings (CPU)
import time

def ensure_hybrid_collection(client: QdrantClient, collection_name: str):
    if client.collection_exists(collection_name):
        print(f"Collection already exists: {collection_name}")
        return

    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size=dense_dim,
                distance=models.Distance.COSINE,
                on_disk=True,
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False)
            )
        },
    )
    print(f"Created hybrid collection: {collection_name}")

ensure_hybrid_collection(qdrant_client, COLLECTION_NAME)

PIPELINE_STATS = {}
all_processed_chunks = []
print(f"Start chunking {len(ds_content_500)} documents...")

t_chunk_start = time.perf_counter()
for row in ds_content_500:
    doc_id = str(row["id"])
    content = row["content"]
    metadata = metadata_by_id.get(doc_id)
    if not metadata:
        continue
    all_processed_chunks.extend(chunker.process_document(content, metadata))
PIPELINE_STATS["chunk_seconds"] = time.perf_counter() - t_chunk_start

print(f"Total chunks: {len(all_processed_chunks)}")

all_texts = [chunk["text_to_embed"] for chunk in all_processed_chunks]

print("Encoding dense vectors with BGE-M3...")
batch_size = 4  # conservative default for CPU
t_dense_start = time.perf_counter()
all_dense_vectors = embedder.encode_dense(all_texts, batch_size=batch_size)
PIPELINE_STATS["dense_seconds"] = time.perf_counter() - t_dense_start

print("Encoding sparse vectors with BGE-M3 lexical weights...")
t_sparse_start = time.perf_counter()
all_sparse_vectors = sparse_encoder.encode_sparse_documents(all_texts, batch_size=batch_size)
PIPELINE_STATS["sparse_seconds"] = time.perf_counter() - t_sparse_start

PIPELINE_STATS["documents"] = len(ds_content_500)
PIPELINE_STATS["chunks"] = len(all_processed_chunks)
print("Dense + sparse vectors ready for hybrid indexing.")
print(PIPELINE_STATS)

In [ ]:
# Cell 7: build payload, upsert to Qdrant, and build payload indexes (Optimized)
import time

points_to_insert = []
t_point_build_start = time.perf_counter()

for idx, chunk in enumerate(all_processed_chunks):
    meta = chunk["metadata"]
    payload = {
        "document_id": str(meta.get("id", "")),
        "chunk_id": chunk["chunk_id"],
        "document_number": meta.get("document_number", "N/A"),
        "title": meta.get("title", "N/A"),
        "legal_type": meta.get("legal_type", "N/A"),
        "legal_sectors": list(meta.get("legal_sectors", meta.get("legal_sectors_list", []))),
        "issuing_authority": meta.get("issuing_authority", "N/A"),
        "signer": meta.get("signer", meta.get("signers", "N/A")),
        "chapter_ref": meta.get("chapter_ref"),
        "article_id": meta.get("article_id"),
        "article_ref": meta.get("article_ref"),
        "article_title": meta.get("article_title"),
        "clause_ref": meta.get("clause_ref"),
        "point_ref": meta.get("point_ref"),
        "reference_tag": chunk["reference_tag"],
        "reference_citation": meta.get("reference_citation"),
        "is_appendix": bool(meta.get("is_appendix", False)),
        "parent_article_text": meta.get("parent_article_text", ""),
        "matched_clause_text": chunk.get("unit_text", ""),
        "chunk_text": chunk["text_to_embed"],
    }

    point = models.PointStruct(
        id=chunk["chunk_id"],
        vector={
            "dense": all_dense_vectors[idx],
            "sparse": all_sparse_vectors[idx],
        },
        payload=payload,
    )
    points_to_insert.append(point)

PIPELINE_STATS["point_build_seconds"] = time.perf_counter() - t_point_build_start

BATCH_UPSERT = 64
print(f"Upserting {len(points_to_insert)} hybrid points to Qdrant Docker...")

t_upsert_start = time.perf_counter()
for start in range(0, len(points_to_insert), BATCH_UPSERT):
    batch = points_to_insert[start : start + BATCH_UPSERT]
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True,
    )
PIPELINE_STATS["upsert_seconds"] = time.perf_counter() - t_upsert_start

print(f"Done upsert {len(points_to_insert)} points to collection {COLLECTION_NAME}.")

def safe_create_payload_index(client, collection_name, field_name, field_schema):
    try:
        client.create_payload_index(
            collection_name=collection_name,
            field_name=field_name,
            field_schema=field_schema,
            wait=True,
        )
        print(f"[index] Created: {field_name}")
    except Exception as exc:
        print(f"[index] Skipped/Exists: {field_name} - {exc}")

print("\n--- Setting up Payload Indexes ---")
# 1. Nhóm Định vị (Exact Match)
for field in ["document_id", "document_number", "chapter_ref", "article_ref", "clause_ref", "point_ref"]:
    safe_create_payload_index(qdrant_client, COLLECTION_NAME, field, models.PayloadSchemaType.KEYWORD)

# 2. Nhóm Phân loại
for field in ["legal_type", "legal_sectors", "issuing_authority", "signer"]:
    safe_create_payload_index(qdrant_client, COLLECTION_NAME, field, models.PayloadSchemaType.KEYWORD)

# 3. Nhóm Cờ trạng thái
safe_create_payload_index(qdrant_client, COLLECTION_NAME, "is_appendix", models.PayloadSchemaType.BOOL)

# 4. Nhóm Tiêu đề (Searchable Text - Vietnamese Optimized)
safe_create_payload_index(
    qdrant_client, 
    COLLECTION_NAME, 
    "title", 
    models.TextIndexParams(
        type="text",
        tokenizer=models.TokenizerType.WORD,
        min_token_len=2,
        max_token_len=20,
        lowercase=True,
    )
)

# 5. Cấu hình Quantization (Nén int8)
try:
    qdrant_client.update_collection(
        collection_name=COLLECTION_NAME,
        quantization_config=models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(
                type="int8",
                quantile=0.99,
                always_ram=True,
            )
        ),
    )
    print("\n[Quantization] Enabled ScalarQuantization int8 for dense vectors.")
except Exception as exc:
    print(f"\n[Quantization] Skip update: {exc}")

total_measured = sum(PIPELINE_STATS.get(k, 0.0) for k in ["chunk_seconds", "dense_seconds", "sparse_seconds", "point_build_seconds", "upsert_seconds"])
PIPELINE_STATS["total_seconds_measured"] = total_measured
print("\nPIPELINE_STATS:", PIPELINE_STATS)


In [ ]:
# CPU runtime estimate based on measured pipeline stats
import math

if not PIPELINE_STATS:
    print("PIPELINE_STATS is empty. Please run Cell 6 and Cell 7 first.")
else:
    measured_docs = max(int(PIPELINE_STATS.get("documents", 0)), 1)
    measured_chunks = max(int(PIPELINE_STATS.get("chunks", 0)), 1)

    t_chunk = float(PIPELINE_STATS.get("chunk_seconds", 0.0))
    t_dense = float(PIPELINE_STATS.get("dense_seconds", 0.0))
    t_sparse = float(PIPELINE_STATS.get("sparse_seconds", 0.0))
    t_point_build = float(PIPELINE_STATS.get("point_build_seconds", 0.0))
    t_upsert = float(PIPELINE_STATS.get("upsert_seconds", 0.0))
    t_total = float(PIPELINE_STATS.get("total_seconds_measured", 0.0))

    sec_per_doc = t_total / measured_docs
    sec_per_chunk = t_total / measured_chunks

    for target_docs in [100, 500, 1000, 10000]:
        est_seconds = sec_per_doc * target_docs
        est_minutes = est_seconds / 60.0
        est_hours = est_minutes / 60.0
        print(
            f"Estimate for {target_docs:>5} docs: "
            f"{est_seconds:,.1f}s | {est_minutes:,.1f}m | {est_hours:,.2f}h"
        )

    print("\nMeasured breakdown on CPU:")
    print(f"- chunking:     {t_chunk:,.2f}s")
    print(f"- dense embed:  {t_dense:,.2f}s")
    print(f"- sparse embed: {t_sparse:,.2f}s")
    print(f"- point build:  {t_point_build:,.2f}s")
    print(f"- upsert:       {t_upsert:,.2f}s")
    print(f"- total:        {t_total:,.2f}s")
    print(f"- avg/doc:      {sec_per_doc:,.3f}s")
    print(f"- avg/chunk:    {sec_per_chunk:,.3f}s")

In [ ]:
# Cell 8: Native Hybrid Legal Retrieval -> Payload Filtering + BGE-M3 + Native RRF
from qdrant_client import models
from typing import List, Optional, Any


def detect_sector_hints(query: str, hot_sectors: List[str]) -> List[str]:
    lowered = query.lower()
    hits = [sector for sector in hot_sectors if sector.lower() in lowered]
    return hits[:3]


class LegalHybridRetriever:
    def __init__(
        self,
        client: QdrantClient,
        collection_name: str,
        hybrid_encoder: Any,
        hot_sectors: List[str],
    ):
        self.client = client
        self.collection_name = collection_name
        self.hybrid_encoder = hybrid_encoder
        self.hot_sectors = hot_sectors

    def build_filter(
        self,
        query: str,
        is_appendix: Optional[bool] = None,
        legal_type: Optional[str] = None,
        doc_number: Optional[str] = None,
    ):
        must_conditions = []
        should_conditions = []

        if is_appendix is not None:
            must_conditions.append(
                models.FieldCondition(
                    key="is_appendix",
                    match=models.MatchValue(value=is_appendix),
                )
            )
        if legal_type:
            must_conditions.append(
                models.FieldCondition(
                    key="legal_type",
                    match=models.MatchValue(value=legal_type),
                )
            )
        if doc_number:
            must_conditions.append(
                models.FieldCondition(
                    key="document_number",
                    match=models.MatchValue(value=doc_number),
                )
            )

        # Sector filter: kiem tra sector co nam trong mang payload legal_sectors hay khong
        for sector in detect_sector_hints(query, self.hot_sectors):
            should_conditions.append(
                models.FieldCondition(
                    key="legal_sectors",
                    match=models.MatchValue(value=sector),
                )
            )
            should_conditions.append(
                models.FieldCondition(
                    key="title",
                    match=models.MatchText(text=sector),
                )
            )

        if not must_conditions and not should_conditions:
            return None

        return models.Filter(
            must=must_conditions or None,
            should=should_conditions or None,
        )

    def search(
        self,
        query: str,
        limit: int = 5,
        is_appendix: Optional[bool] = None,
        legal_type: Optional[str] = None,
        doc_number: Optional[str] = None,
    ):
        query_filter = self.build_filter(
            query=query,
            is_appendix=is_appendix,
            legal_type=legal_type,
            doc_number=doc_number,
        )

        dense_query = self.hybrid_encoder.encode_dense([query], batch_size=1)[0]
        sparse_query = self.hybrid_encoder.encode_query_sparse(query)

        prefetch_limit = max(limit * 3, 10)

        hits = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                models.Prefetch(
                    query=dense_query,
                    using="dense",
                    limit=prefetch_limit,
                    filter=query_filter,
                ),
                models.Prefetch(
                    query=sparse_query,
                    using="sparse",
                    limit=prefetch_limit,
                    filter=query_filter,
                )
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=max(limit * 2, 10),
            with_payload=True,
        ).points

        dedup_results = []
        seen_articles = set()

        for hit in hits:
            payload = hit.payload
            article_id = payload.get("article_id") or payload.get("chunk_id")
            if article_id in seen_articles:
                continue
            seen_articles.add(article_id)

            dedup_results.append({
                "id": hit.id,
                "payload": payload,
                "rrf_score": hit.score,
                "dense_score": "Native Fusion",
                "sparse_score": "Native Fusion",
            })
            if len(dedup_results) >= limit:
                break

        return dedup_results


def print_retrieval_results(results):
    if not results:
        print("Khong tim thay ket qua phu hop.")
        return

    for idx, item in enumerate(results, start=1):
        payload = item["payload"]
        matched_clause = (payload.get("matched_clause_text") or "").replace("\n", " ")
        parent_article = (payload.get("parent_article_text") or "").replace("\n", " ")
        sectors = payload.get("legal_sectors") or []
        print(
            f"[{idx}] RRF_Score={item['rrf_score']:.4f} | Qdrant Native Fusion"
        )
        print(f"    Title: {payload.get('title')}")
        print(f"    Citation: {payload.get('reference_citation')}")
        print(f"    Appendix: {payload.get('is_appendix')} | Sectors: {sectors}")
        print(f"    Matched clause: {matched_clause[:260]}...")
        print(f"    Full article preview: {parent_article[:260]}...")
        print("-" * 140)


retriever = LegalHybridRetriever(
    client=index_client,
    collection_name=COLLECTION_NAME,
    hybrid_encoder=hybrid_encoder,
    hot_sectors=HOT_SECTORS,
)


def legal_retrieval(
    query: str,
    limit: int = 5,
    is_appendix: Optional[bool] = None,
    legal_type: Optional[str] = None,
    doc_number: Optional[str] = None,
):
    print(f"TRUY VAN: {query}")
    results = retriever.search(
        query=query,
        limit=limit,
        is_appendix=is_appendix,
        legal_type=legal_type,
        doc_number=doc_number,
    )
    print_retrieval_results(results)
    return results

In [ ]:
"""
LLM adapter for notebook, reusing core.llm and .env settings.
"""
from collections import defaultdict
from typing import List, Dict
from core.llm import chat_completion


def repo_chat_completion(
    messages: List[Dict[str, str]],
    temperature: float = 0.3,
    provider: str = None,
    model: str = None,
) -> str:
    return chat_completion(
        messages=messages,
        temperature=temperature,
        provider=provider,
        model=model,
    )

In [ ]:
# Cell 9: LLM wrapper + memory 5 turn + rewrite query + augment/generation
class SessionMemory:
    def __init__(self, max_turns: int = 5):
        self.max_turns = max_turns
        self.sessions = defaultdict(list)

    def get_history(self, session_id: str):
        return self.sessions.get(session_id, [])

    def add_message(self, session_id: str, role: str, content: str):
        self.sessions[session_id].append({"role": role, "content": content})
        max_messages = self.max_turns * 2
        if len(self.sessions[session_id]) > max_messages:
            self.sessions[session_id] = self.sessions[session_id][-max_messages:]


class NotebookLegalLLM:
    def __init__(self, provider: str = "mock", model_name: Optional[str] = None):
        self.provider = provider
        self.model_name = model_name or os.getenv(
            "NOTEBOOK_LLM_MODEL",
            "meta-llama/Meta-Llama-3.1-8B-Instruct",
        )
        self.generator = None

    def _load_local_generator(self):
        if self.generator is None:
            import torch
            from transformers import pipeline

            torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
            self.generator = pipeline(
                "text-generation",
                model=self.model_name,
                device_map="auto" if torch.cuda.is_available() else None,
                torch_dtype=torch_dtype,
            )
        return self.generator

    def generate_text(
        self,
        messages: List[Dict[str, str]],
        temperature: float = 0.2,
        max_new_tokens: int = 512,
    ) -> str:
        if self.provider == "repo_api" and repo_chat_completion is not None:
            return repo_chat_completion(messages=messages, temperature=temperature)

        if self.provider == "local_llama":
            generator = self._load_local_generator()
            prompt = "\n\n".join(
                [f"{msg['role'].upper()}: {msg['content']}" for msg in messages]
            ) + "\nASSISTANT:"
            output = generator(
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0,
                temperature=max(temperature, 0.1),
                return_full_text=False,
            )[0]["generated_text"]
            return output.strip()

        return ""

    def rewrite_query(self, history: List[Dict[str, str]], current_query: str) -> str:
        if not history:
            return current_query

        if self.provider in {"repo_api", "local_llama"}:
            history_text = "\n".join(
                [f"{message['role']}: {message['content']}" for message in history]
            )
            rewrite_prompt = (
                "Viet lai cau hoi hien tai thanh mot standalone query phap ly bang tieng Viet.\n"
                f"Lich su hoi dap:\n{history_text}\n\n"
                f"Cau hoi hien tai: {current_query}\n\n"
                "Chi tra ve cau query da viet lai."
            )
            rewritten = self.generate_text(
                [{"role": "user", "content": rewrite_prompt}],
                temperature=0.0,
                max_new_tokens=128,
            )
            return rewritten or current_query

        recent_user_turns = [m["content"] for m in history if m["role"] == "user"][-2:]
        if recent_user_turns:
            return " | ".join(recent_user_turns + [current_query])
        return current_query

    def answer_from_context(
        self,
        user_query: str,
        rewritten_query: str,
        mode: str,
        results: List[Dict[str, Any]],
    ) -> str:
        if not results:
            return "Chua tim thay ngu canh phu hop trong kho du lieu de tra loi cau hoi nay."

        if self.provider in {"repo_api", "local_llama"}:
            context_blocks = []
            for idx, item in enumerate(results[:4], start=1):
                payload = item["payload"]
                context_blocks.append(
                    f"[{idx}] {payload.get('reference_citation')}\n"
                    f"Title: {payload.get('title')}\n"
                    f"Full article:\n{payload.get('parent_article_text')}\n"
                    f"Matched clause:\n{payload.get('matched_clause_text')}\n"
                )

            system_prompt = (
                "Ban la tro ly phap ly Viet Nam. "
                "Chi duoc dua tren CONTEXT va phai trich dan dieu-khoan-diem neu co."
            )
            user_prompt = (
                f"[MODE]: {mode}\n"
                f"[QUERY_GOC]: {user_query}\n"
                f"[QUERY_DA_VIET_LAI]: {rewritten_query}\n\n"
                f"[CONTEXT]\n{chr(10).join(context_blocks)}\n\n"
                "Tra loi ro rang, ngan gon, va tach rieng muc Can cu phap ly."
            )
            generated = self.generate_text(
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
                max_new_tokens=512,
            )
            if generated:
                return generated

        if mode == "sector_search":
            lines = [
                f"Tim thay {len(results)} ket qua lien quan den truy van: {rewritten_query}",
                "",
            ]
            for idx, item in enumerate(results[:5], start=1):
                payload = item["payload"]
                lines.append(
                    f"{idx}. {payload.get('title')} | {payload.get('reference_citation')} | "
                    f"Linh vuc: {payload.get('legal_sectors', [])}"
                )
            return "\n".join(lines)

        top_payload = results[0]["payload"]
        lines = [
            f"Tra loi so bo cho cau hoi: {user_query}",
            "",
            (top_payload.get("matched_clause_text") or top_payload.get("parent_article_text", ""))[:900],
            "",
            "Can cu phap ly:",
        ]
        for item in results[:3]:
            payload = item["payload"]
            lines.append(f"- {payload.get('reference_citation')}")
        return "\n".join(lines)


class LegalChatAssistant:
    def __init__(self, retriever: LegalHybridRetriever, llm: NotebookLegalLLM, max_turns: int = 5):
        self.retriever = retriever
        self.llm = llm
        self.memory = SessionMemory(max_turns=max_turns)

    def ask(
        self,
        session_id: str,
        query: str,
        mode: str = "knowledge_qa",
        limit: int = 5,
        is_appendix: Optional[bool] = None,
        legal_type: Optional[str] = None,
        doc_number: Optional[str] = None,
    ):
        history = self.memory.get_history(session_id)
        rewritten_query = self.llm.rewrite_query(history, query)
        retrieval_results = self.retriever.search(
            query=rewritten_query,
            limit=limit,
            is_appendix=is_appendix,
            legal_type=legal_type,
            doc_number=doc_number,
        )
        answer = self.llm.answer_from_context(
            user_query=query,
            rewritten_query=rewritten_query,
            mode=mode,
            results=retrieval_results,
        )

        self.memory.add_message(session_id, "user", query)
        self.memory.add_message(session_id, "assistant", answer)

        return {
            "answer": answer,
            "rewritten_query": rewritten_query,
            "results": retrieval_results,
        }


NOTEBOOK_LLM_PROVIDER = os.getenv("NOTEBOOK_LLM_PROVIDER", "mock")
legal_llm = NotebookLegalLLM(provider=NOTEBOOK_LLM_PROVIDER)
assistant = LegalChatAssistant(retriever=retriever, llm=legal_llm, max_turns=5)

print(f"Notebook LLM provider: {NOTEBOOK_LLM_PROVIDER}")

In [ ]:
# Cell 10: demo retrieval + demo chat memory 5 turn cho 2 kieu cau hoi phap ly
print("=" * 140)
print("DEMO 1 - smart legal retrieval")
print("=" * 140)
legal_retrieval(
    query="Tim cac van ban lien quan den linh vuc giao thong duong bo",
    limit=5,
    is_appendix=None,
)

print("\n" + "=" * 140)
print("DEMO 2 - tim trong phu luc")
print("=" * 140)
legal_retrieval(
    query="Ma ho so 1.006639 la thu tuc gi?",
    limit=5,
    is_appendix=True,
)

print("\n" + "=" * 140)
print("DEMO 3 - hoi dap + memory 5 turn")
print("=" * 140)

demo_questions = [
    ("knowledge_qa", "Quy dinh nao noi ve hieu luc thi hanh cua van ban nay?", None),
    ("knowledge_qa", "Van ban moi hon trong nhom nay co thay the gi khong?", None),
    ("sector_search", "Tim van ban lien quan den dat dai trong tap mau", None),
]

for mode, question, appendix_flag in demo_questions:
    response = assistant.ask(
        session_id="demo-session-1",
        query=question,
        mode=mode,
        limit=4,
        is_appendix=appendix_flag,
    )
    print(f"\nQuestion: {question}")
    print(f"Rewritten: {response['rewritten_query']}")
    print(response["answer"])
    print("Top citations:")
    for item in response["results"][:3]:
        print(f"- {item['payload'].get('reference_citation')}")